In [ ]:
from datetime import time
import re


def time_to_seconds(t: time) -> int:
    return t.hour * 3600 + t.minute * 60 + t.second


def seconds_to_time(s: int) -> time:
    h, remainder = divmod(s, 3600)
    m, s = divmod(remainder, 60)
    return time(h % 24, m, s)


def convert_to_24_hour_time(time_to_normalize: str) -> time:
    match: re.Match[str] | None = re.match(
        r"(\d{2}):(\d{2}):(\d{2})", time_to_normalize)
    if not match:
        raise ValueError(f"Invalid time format: {time_to_normalize}")

    hour, minute, second = map(int, match.groups())

    if hour >= 24:
        hour -= 24

    return time(hour, minute, second)


In [ ]:
from dataclasses import dataclass
from datetime import time
from geopy.point import Point
import utils
from collections import defaultdict
import sys
import enum


@dataclass
class Node:
    name: str
    location: Point
    _hash: int = 0

    def __post_init__(self) -> None:
        self._hash = hash((self.name, self.location.format_unicode()))

    def __hash__(self) -> int:
        return self._hash


@dataclass
class CommunicationStep:
    company: str
    line: str
    departure_time: time
    arrival_time: time
    start_stop: Node
    end_stop: Node

    @staticmethod
    def from_parsed_csv_line(row: list[str]) -> "CommunicationStep":
        start_stop_point: Point = Point(latitude=row[7], longitude=row[8])
        start_stop = Node(row[5], start_stop_point)

        end_stop_point: Point = Point(latitude=row[9], longitude=row[10])
        end_stop = Node(row[6], end_stop_point)

        company, line, departure_str, arrival_str = row[1:5]

        departure_time: time = utils.convert_to_24_hour_time(departure_str)
        arrival_time: time = utils.convert_to_24_hour_time(arrival_str)

        new_communication_step = CommunicationStep(
            company, line, departure_time, arrival_time, start_stop, end_stop)

        return new_communication_step

    def __str__(self):
        return f"line {self.line} | {self.start_stop} {self.departure_time} -> {self.end_stop} {self.arrival_time}"

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, CommunicationStep):
            return False
        return (
            self.company == other.company and
            self.line == other.line and
            self.departure_time == other.departure_time and
            self.arrival_time == other.arrival_time and
            self.start_stop == other.start_stop and
            self.end_stop == other.end_stop
        )

    def __hash__(self) -> int:
        return hash((self.company, self.line, self.departure_time, self.arrival_time, self.start_stop, self.end_stop))


class Graph:
    def __init__(self) -> None:
        self.nodes: dict[str, Node] = {}
        self.edges: dict[tuple[str, str],
                         set[CommunicationStep]] = defaultdict(set)
        self.adjacency_list: dict[str,
                                  set[CommunicationStep]] = defaultdict(set)


@dataclass
class LineStep:
    start_node_name: str
    end_node_name: str
    line: str
    start_time: time
    end_time: time


@dataclass
class Path:
    steps: list[LineStep]
    cost: float
    calculation_time: float

    def pretty_print(self) -> None:
        print("Schedule:")
        for step in self.steps:
            print(
                f"{step.line}\t| {step.start_node_name} {step.start_time} -> {step.end_node_name} {step.end_time}")
        print(f"Total cost: {self.cost} units", file=sys.stderr, flush=True)
        print(
            f"Execution time: {self.calculation_time:.4f} seconds", file=sys.stderr, flush=True)


class NoPathFoundError(Exception):
    """Raised when no path is found between the start and end nodes."""
    pass


class OptimizationCriterion(enum.Enum):
    TIME = "time"
    TRANSFERS = "transfers"


In [ ]:
from geopy.distance import geodesic
import models
from functools import lru_cache
from datetime import time


@lru_cache(maxsize=None)
def distance_heuristic(node1: models.Node, node2: models.Node) -> float:
    return geodesic(node1.location, node2.location).kilometers ** 2


def transfer_heuristic(transfer_count: int) -> float:
    transfer_penalty_weight = 500000
    return transfer_count * transfer_penalty_weight


def clear_caches() -> None:
    distance_heuristic.cache_clear()


def post_clear_cache(func):
    def wrapper(*args, **kwargs) -> None:
        result = func(*args, **kwargs)
        clear_caches()
        return result
    return wrapper


def generate_path(start: str, path_taken: list[models.CommunicationStep], elapsed: float, cost: float) -> models.Path:
    path_steps: list[models.LineStep] = []

    none_time: time = time(0, 0)

    step: models.CommunicationStep = path_taken[0]
    last_line: str = step.line
    path_steps.append(models.LineStep(
        start, "", step.line, step.departure_time, none_time))

    for i, step in enumerate(path_taken[:-1]):
        if step.line == last_line:
            continue
        last_line = step.line

        path_steps[-1].end_time = path_taken[i - 1].arrival_time
        path_steps[-1].end_node_name = step.start_stop.name

        path_steps.append(models.LineStep(step.start_stop.name,
                          "", last_line, step.departure_time, none_time))

    step = path_taken[-1]
    path_steps[-1].end_time = step.arrival_time
    path_steps[-1].end_node_name = step.end_stop.name

    path: models.Path = models.Path(path_steps, cost, elapsed)
    return path


In [ ]:
from dataclasses import dataclass, field
from datetime import time
from utils import time_to_seconds
from datetime import datetime
import heapq
from itertools import count
import time as t
from models import Node, Path, OptimizationCriterion, CommunicationStep, NoPathFoundError, Graph
from path_utils import generate_path, post_clear_cache


@dataclass(order=True)
class QueueEntry:
    priority: int
    counter: int
    current_stop_name: str = field(compare=False)
    path_taken: list[CommunicationStep] = field(compare=False)
    current_time_sec: int = field(compare=False)


@post_clear_cache
def dijkstra(start: str, end: str, start_time: time, graph: Graph) -> Path:
    if start not in graph.nodes:
        raise ValueError("Start stop does not exist in the graph.")
    if end not in graph.nodes:
        raise ValueError("End stop does not exist in the graph.")

    start_node: Node = graph.nodes[start]
    end_node: Node = graph.nodes[end]

    start_time_sec: int = time_to_seconds(start_time)

    queue: list[QueueEntry] = []
    counter = count()
    heapq.heappush(queue, QueueEntry(0, next(counter),
                   start_node.name, [], start_time_sec))

    visited: set[str] = set()

    start_time_perf: float = t.perf_counter()

    while queue:
        entry: QueueEntry = heapq.heappop(queue)

        if entry.current_stop_name in visited:
            continue
        visited.add(entry.current_stop_name)

        if entry.current_stop_name == end_node.name:
            elapsed: float = t.perf_counter() - start_time_perf

            path: Path = generate_path(
                start, entry.path_taken, elapsed, entry.priority)

            return path

        for (start_name, end_name), steps in graph.edges.items():
            if start_name != entry.current_stop_name:
                continue
            for step in steps:
                departure_seconds: int = time_to_seconds(step.departure_time)
                arrival_seconds: int = time_to_seconds(step.arrival_time)

                if departure_seconds >= entry.current_time_sec:
                    travel_time: int = arrival_seconds - entry.current_time_sec
                    if travel_time < 0:
                        travel_time += 86400  # handle crossing midnight

                    new_priority: int = entry.priority + travel_time
                    new_queue_entry = QueueEntry(
                        new_priority,
                        next(counter),
                        end_name,
                        entry.path_taken + [step],
                        arrival_seconds
                    )
                    heapq.heappush(queue, new_queue_entry)

    raise NoPathFoundError(f"No path found from {start} to {end}")
